# Introduction

## Imports

In [1]:
import io
import pandas as pd
import chess
import chess.pgn
import chess.engine
from tqdm import tqdm
import math
import os 
import subprocess
import re
import numpy as np
import pymc as pm
import arviz as az
import joblib

from pymc.sampling.jax import sample_numpyro_nuts

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

## Feature Engeneering

### 1) Restructuring Dataset to Single Plies + Objective Chess Features

Here the datasaet is unraveld into singular plys. For every ply objective chess features are added from the `pyhton-chess` library.
Not all featured are used for the target model, rather for further feature Engeneering.

| Feature | Meaning | Model Feature |
|---|---|---|
| `game_id` | ID of the original game from which the ply was extracted. (Index number of row) | No |
| `ply` | Ply number within the game. | Yes |
| `move_number` | Full move number in the game. | Yes |
| `fen_before` | FEN representation of the board position before the played move. | No |
| `move_uci` | Played move in UCI notation, e.g. `e2e4`. | No |
| `move_san` | Played move in SAN notation, e.g. `Nf3`, `exd5`, or `O-O`. | No |
| `player_elo` | Elo rating of the player who made that move. | Yes |
| `opponent_elo` | Elo rating of the opponent. | Yes |
| `result` | Game result from the original CSV file. | Optional / No |
| `label` | Target variable: $0$ for human-like/Maia moves and $1$ for Stockfish/engine moves. | Target |
| `source` | Sample source of the ply:`lichess_opening`, `maia`, or `stockfish`. | No |
| `is_capture` | Indicates whether the move captures a piece. $1$ for yes, $0$ for no. | Yes |
| `gives_check` | Indicates whether the move gives check. $1$ for yes, $0$ for no. | Yes |
| `is_castling` | Indicates if the move is castling. $1$ for yes, $0$ for no. | Yes |
| `is_promotion` | Indicates if the move is a pawn promotion. $1$ for yes, $0$ for no. | Yes |
| `legal_moves_before` | Number of legal moves possible before the played move. | Yes |
| `material_balance` | Material balance from the perspective of the player of the move. (positive number means advantage, negative means disadvantage | Yes |
| `piece_none` | One-hot feature indicating move was not made by known piece. For legal moves this should always be $0$. | Yes |
| `piece_pawn` | One-hot feature indicating that the moved piece is a pawn. | Yes |
| `piece_knight` | One-hot feature indicating that the moved piece is a knight. | Yes |
| `piece_bishop` | One-hot feature indicating that the moved piece is a bishop. | Yes |
| `piece_rook` | One-hot feature indicating that the moved piece is a rook. | Yes |
| `piece_queen` | One-hot feature indicating that the moved piece is a queen. | Yes |
| `piece_king` | One-hot feature indicating that the moved piece is a king. | Yes |
| `captured_none` | One-hot feature indicating that no piece was captured. | Yes |
| `captured_pawn` | One-hot feature indicating that a pawn was captured. | Yes |
| `captured_knight` | One-hot feature indicating that a knight was captured. | Yes |
| `captured_bishop` | One-hot feature indicating that a bishop was captured. | Yes |
| `captured_rook` | One-hot feature indicating that a rook was captured. | Yes |
| `captured_queen` | One-hot feature indicating that a queen was captured. | Yes |
| `captured_king` | One-hot feature indicating that a king was captured. (Obviously should be always $0$ as well) | Yes |
| `color_white` | One-hot feature for the color of move. | Yes |
| `color_black` | One-hot feature for the color of move. | Yes |

In [4]:
INPUT_CSV = "Games.csv"
OUTPUT_CSV = "moves.csv"


PIECE_VALUES = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
    chess.KING: 0,
}




df = pd.read_csv(INPUT_CSV, dtype={
    "Liste cheat white": str,
    "Liste cheat black": str,
})
all_ply_rows = []


def material_score(board, color):
    score = 0
    for piece_type, value in PIECE_VALUES.items():
        score += len(board.pieces(piece_type, color)) * value
    return score

def pgn_to_plies(row, game_id):

    ply_rows = []


    white_bits = row["Liste cheat white"]
    black_bits = row["Liste cheat black"]
    pgn_text = row["Game"]

    game = chess.pgn.read_game(io.StringIO(pgn_text))
    if game is None:
        return []

    board = game.board()
 

    white_move_i = 0
    black_move_i = 0

    for ply, move in enumerate(game.mainline_moves(), start=1):

        player_color = board.turn
        opponent_color = not player_color
        fen_before = board.fen() # always the same before first move
        move_san = board.san(move)
        move_uci = move.uci()

        player_material = material_score(board, player_color)
        opponent_material = material_score(board, opponent_color)
        material_balance = player_material - opponent_material

        player_piece = board.piece_at(move.from_square)
        player_piece_type_id = player_piece.piece_type if player_piece else 0

        if board.is_en_passant(move):
            captured_piece_type_id = chess.PAWN
        else:
            captured_piece = board.piece_at(move.to_square)
            captured_piece_type_id = captured_piece.piece_type if captured_piece else 0
        
        is_capture = int(board.is_capture(move))
        gives_check = int(board.gives_check(move))
        is_casteling = int(board.is_castling(move))
        is_promotion = int(move.promotion is not None)
        legal_moves_before = board.legal_moves.count()

        if player_color == chess.WHITE:
            player_elo = row["Elo White"]
            opponent_elo = row["Elo Black"]

            if white_move_i < 10:   # first 10 moves are always labeled as 0 (lichess opening samples)
                label = 0
                source = "lichess_opening"
            else:
                bit_i = white_move_i - 10
                if bit_i >= len(white_bits):
                    break

                label = int(white_bits[bit_i])
                source = "stockfish" if label == 1 else "maia"

            white_move_i += 1

        else:
            player_elo = row["Elo Black"]
            opponent_elo = row["Elo White"]

            if black_move_i < 10:
                label = 0
                source = "lichess_opening"
            else:
                bit_i = black_move_i - 10
                if bit_i >= len(black_bits):
                    break

                label = int(black_bits[bit_i])
                source = "stockfish" if label == 1 else "maia"

            black_move_i += 1

        ply_rows.append({
            "game_id": game_id,
            "ply": ply,
            "color_white": int(player_color == chess.WHITE),
            "color_black": int(player_color == chess.BLACK),
            "fen_before": fen_before,
            "move_san": move_san,
            "move_uci": move_uci,
            "move_number": board.fullmove_number,
            "player_elo": player_elo,
            "opponent_elo": opponent_elo,
            "material_balance": material_balance,
            "piece_none": int(player_piece_type_id == 0),
            "piece_pawn": int(player_piece_type_id == chess.PAWN),
            "piece_knight": int(player_piece_type_id == chess.KNIGHT),
            "piece_bishop": int(player_piece_type_id == chess.BISHOP),
            "piece_rook": int(player_piece_type_id == chess.ROOK),
            "piece_queen": int(player_piece_type_id == chess.QUEEN),
            "piece_king": int(player_piece_type_id == chess.KING),
            "captured_none": int(captured_piece_type_id == 0),
            "captured_pawn": int(captured_piece_type_id == chess.PAWN),
            "captured_knight": int(captured_piece_type_id == chess.KNIGHT),
            "captured_bishop": int(captured_piece_type_id == chess.BISHOP),
            "captured_rook": int(captured_piece_type_id == chess.ROOK),
            "captured_queen": int(captured_piece_type_id == chess.QUEEN),
            "captured_king": int(captured_piece_type_id == chess.KING),
            "is_capture": is_capture,
            "gives_check": gives_check,
            "is_casteling": is_casteling,
            "is_promotion": is_promotion,
            "legal_moves_before": legal_moves_before,
            "source": source,
            "result": row["Score"],
            "label": label,
        })

        board.push(move) # make the move on the board for the next iteration

    return ply_rows


for game_id, row in tqdm(df.iterrows(), total=len(df), desc="Extracting plies"):
    ply_rows = pgn_to_plies(row, game_id)
    all_ply_rows.extend(ply_rows)

ply_df = pd.DataFrame(all_ply_rows)
ply_df.to_csv(OUTPUT_CSV, index=False)




Extracting plies: 100%|█████████████████| 48932/48932 [04:41<00:00, 173.73it/s]


### 2) Adding Stockfish-based Engine Features

Since the model is supposed to learn to distinguish between human-like moves and engine moves, features are added that directly compare the played move to Stockfish's evaluation. These features can capture how closely the played move aligns with Stockfish's top choices.
Following features are added by running Stockfish on the position before the move.

| Feature | Meaning | Model Feature |
|---|---|---|
| `sf_cp_loss` | Difference of score between stockfish's best move and played move in *centipawns*: $\max(0, cp_{best} - cp_{played})$ - this is capped to 1000 since mate moves can produce enormous cp losses| No |
| `sf_cp_loss_log` | Log-transformed centipawn loss: $\log(1 + \text{sf\_cp\_loss})$. This reduces the influence of very large centipawn losses while preserving differences between strong moves. | Yes |
| `sf_rank_topk` | Rank of the played move within Stockfish's top-$k$ candidate moves. If the move is not among the top-$k$ moves, the rank is set to $k + 1$. | Yes |
| `sf_top1` | Indicates whether the played move is exactly Stockfish's top move. $1$ if yes, $0$ otherwise. | Yes |

In [ ]:
STOCKFISH_PATH = "/usr/games/stockfish"

DEPTH = 20
MULTIPV = 5 # how many top moves Stockfish should return for each position

sf_cp_losses = []
sf_cp_loss_logs = []
sf_rank_topks = []
sf_top1s = []



def score_to_cp(score, color):
    return score.pov(color).score(mate_score=100000)

with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:

    sf_cp_loss = None
    sf_cp_loss_log = None
    sf_rank_topk = None
    sf_top1 = None
    for row in tqdm(ply_df.itertuples(), total=len(ply_df), desc="Adding Stockfish features"):
        board = chess.Board(row.fen_before)
        played_move = chess.Move.from_uci(row.move_uci)

        color = board.turn

        # stockfish analysis before move played
        infos = engine.analyse(board, chess.engine.Limit(depth=DEPTH), multipv=MULTIPV)
        top_moves = []

        for info in infos: # info struct for each top ranked move
            pv = info.get("pv")
            if not pv or "score" not in info:
                continue

            candidate_move = pv[0] # we only care about first move in PV (principal variation)
            candidate_score = score_to_cp(info["score"], color)
            top_moves.append((candidate_move, candidate_score))

        if not top_moves:
            sf_cp_loss_logs.append(np.log1p(1000)) # if Stockfish fails to return a move, assign max loss
            sf_rank_topks.append(MULTIPV + 1) # if Stockfish fails to return a move, assign max rank
            sf_top1s.append(0) 
            continue

        best_move, best_score = top_moves[0]

        sf_top1 = int(played_move == best_move)
        sf_top1s.append(sf_top1)

        sf_rank_topk = MULTIPV + 1 # default rank if not found in top moves
        for rank, (candidate_move, candidate_score) in enumerate(top_moves, start=1):
            if played_move == candidate_move:
                sf_rank_topk = rank
                break
        sf_rank_topks.append(sf_rank_topk)

        # stockfish analysis after move played 
        played_info = engine.analyse(
            board,
            chess.engine.Limit(depth=DEPTH),
            root_moves=[played_move]
        )

        played_cp = score_to_cp(played_info["score"], color)
        sf_cp_loss = max(0, best_score - played_cp) # avoid negative loss if played move is better than Stockfish's best move
        sf_cp_loss_clean = min(sf_cp_loss, 1000) # cap loss to [0, 1000] to reduce influence of extreme values of mate moves 
        sf_cp_losses.append(sf_cp_loss_clean)
        sf_cp_loss_log = math.log1p(sf_cp_loss_clean) 
        sf_cp_loss_logs.append(sf_cp_loss_log)

ply_df["sf_cp_loss"] = sf_cp_losses
ply_df["sf_cp_loss_log"] = sf_cp_loss_logs
ply_df["sf_rank_topk"] = sf_rank_topks
ply_df["sf_top1"] = sf_top1s

        

NOTE: Since running that many stockfish instances for such amount of plies, an optimized script was created for this procedure, in order to use parallel multithreading on my CPU.

Execute `stockfish_parallel.py`.


For further aproach result of parallel exectution is loaded from a csv-file:

In [3]:
SF_DATA_CSV = "moves_sf.csv"
sf_df = pd.read_csv(SF_DATA_CSV, dtype={
    "Liste cheat white": str,
    "Liste cheat black": str,
})

FileNotFoundError: [Errno 2] No such file or directory: 'moves_sf.csv'

### 3) Adding human-likelyness Features (Maia)

Unfortunately `lc0` can't be used within `chess-python`, therefore we need to crate a bash-subprocess fetching Maia-stats for each ply.

Methods for executing and fetching Maia-features:

Methods to interact with lc0-Maia instance:

In [4]:
LC0_PATH = os.path.expanduser("~/.local/bin/lc0")
MAIA_WEIGHTS_DIR = os.path.expanduser("~/Games/maia-chess/maia_weights")


def send(proc, cmd):
    proc.stdin.write(cmd + "\n")
    proc.stdin.flush()


def read_until(proc, marker):
    lines = []

    while True:
        line = proc.stdout.readline()

        if not line:
            break

        line = line.rstrip()
        lines.append(line)

        if marker in line:
            break

    return lines


def start_maia_process(weights_path):
    proc = subprocess.Popen(
        [
            LC0_PATH,
            f"--weights={weights_path}",
            "--verbose-move-stats",
            "--threads=1",
        ],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    send(proc, "uci")
    read_until(proc, "uciok")

    send(proc, "isready")
    read_until(proc, "readyok")

    return proc


def stop_maia_process(proc):
    try:
        send(proc, "quit")
    except Exception:
        pass

    try:
        proc.terminate()
    except Exception:
        pass


def get_maia_prob_for_move(proc, fen, move_uci):
    send(proc, f"position fen {fen}")
    send(proc, "go nodes 1")

    pattern = re.compile(
        rf"info string\s+{re.escape(move_uci)}\s+.*?\(P:\s*([0-9.]+)%\)"
    )

    prob = None

    while True:
        line = proc.stdout.readline()

        if not line:
            break

        line = line.rstrip()

        match = pattern.search(line)
        if match:
            prob = float(match.group(1)) / 100.0

        if line.startswith("bestmove"):
            break

    return prob



Following features are added.

| Feature | Meaning | Model Feature |
|---|---|---|
| `human_prob` | Propability of played move being human-like according to  `Maia` | Yes |

In [ ]:
ELO_WEIGHTS = [1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900]

sf_maia_df = sf_df.copy()

sf_maia_df["elo_group"] = 0
sf_maia_df["maia_prob"] = -1.0

# group each ply by elo group to not start lc0 on every move
for idx, row in tqdm(sf_maia_df.iterrows(), total=len(sf_maia_df), desc="Grouping plies by ELO"):
    player_elo = row["player_elo"]

    if pd.isna(player_elo) or int(player_elo) < 0:
        sf_maia_df.at[idx, "elo_group"] = 1500
        continue

    player_elo = int(player_elo)

    closest = min(
        ELO_WEIGHTS,
        key=lambda e: abs(e - player_elo)
    )

    sf_maia_df.at[idx, "elo_group"] = closest


def maia_weights_path(maia_elo):
    return os.path.join(MAIA_WEIGHTS_DIR, f"maia-{maia_elo}.pb.gz")


for maia_elo, group in sf_maia_df.groupby("elo_group"):
    print(f"Processing ELO group {maia_elo} with {len(group)} plies")

    weights_path = maia_weights_path(maia_elo)

    if not os.path.exists(weights_path):
        print(f"Warning: Weights file not found for ELO {maia_elo} at path {weights_path}. Skipping this group.")
        continue

    proc = start_maia_process(weights_path)

    try:
        for row in tqdm(group.itertuples(), total=len(group), desc=f"Processing plies for ELO {maia_elo}"):
            fen = row.fen_before
            move_uci = row.move_uci

            maia_prob = get_maia_prob_for_move(proc, fen, move_uci)

            sf_maia_df.at[row.Index, "human_prob"] = maia_prob

    finally:
        stop_maia_process(proc)


sf_maia_df.to_csv("moves_sf_maia.csv", index=False)

Grouping plies by ELO: 100%|██████████| 1585603/1585603 [01:24<00:00, 18718.96it/s]


Processing ELO group 1100 with 29255 plies


Processing plies for ELO 1100: 100%|██████████| 29255/29255 [00:26<00:00, 1087.56it/s]


Processing ELO group 1200 with 59778 plies


Processing plies for ELO 1200: 100%|██████████| 59778/59778 [00:57<00:00, 1048.14it/s]


...

Processing plies for ELO 1700: 100%|██████████| 237596/237596 [03:57<00:00, 998.96it/s] 


Processing ELO group 1800 with 294403 plies


Processing plies for ELO 1800: 100%|██████████| 294403/294403 [04:57<00:00, 989.76it/s] 


Processing ELO group 1900 with 517128 plies


Processing plies for ELO 1900: 100%|██████████| 517128/517128 [08:47<00:00, 980.18it/s] 


### 4) Adding cheat-likeliness Features 

Now we want to introduce a scale that takes into account both human-likelyness and engine-agreement.
For example there are moves that may be stockfish's top choice, but only because they are simply obvious in certain position or well-known theoretical moves, such as capturing a free piece or castling.
These moves should result in low cp_loss, but also high humanlikelyness-probabilites.

Moves with high cp_loss and low humanlikeliness-probabilites are just uncommon moves, which are also possible in a chess game.

We are specially interessted in moves that are marked with low cp_loss (high engine agreement) but low Maia probability, as they indicate the help of chess engines.


Therefore the normalized feature 
$$
\text{human\_surprise} = -\log(\text{human\_prob})
$$
is introduced. The idea is to receive a larger penalty for unlikely human moves:
- if Maia assigns a high probability to a move, then $-\log(p)$ is small
- if Maia assigns a low probability to a move, then $-\log(p)$ becomes large
- very small probabilities are penalized strongly
A move with $0.01$ human-probability is more surprising then a move with $0.8$ probability.


As a surprising move can also arise from just uncommon moves or blunders, we want to count in a normalized scale for engine-agreement 
as well:


$$
\text{engine\_agreement} = \frac{1}{1 + \frac{\max(0, \text{cp\_loss})}{100}}
$$

where $\text{cp\_loss}$ is the centipawn loss of the played move compared to Stockfish's best move from the features so far.


This heuristic maps centipawn loss to a bounded score between 0 and 1:

- if $\text{cp\_loss} = 0$, then $\text{engine\_agreement} = 1$
- if $\text{cp\_loss} = 100$, then $\text{engine\_agreement} = 0.5$
- if $\text{cp\_loss}$ becomes large, the score approaches 0

Division by $100$ is choses because it represents the loss of one pawn, halving the agreement score.


Finally we multiply these scales to create a score for cheat-likeliness:

$$
\text{suspiciousness} = \text{human\_surprise} \cdot \text{engine\_agreement}
$$

We see how obvious moves can balance out suspiciousness e.g.: $\text{human\_surprise} = 0.0001$ and $\text{engine\_agreement} = 1$
still leads to low suspiciousness: $0.0001$

Added features:


| Feature | Meaning | Model Feature |
|---|---|---|
| `human_surprise` | Score for how surprising / uncommon this ply was accroding to Maia | No |
| `engine_agreement` | heuristic to scale how engine-like ply was by utilizing `cp_loss`| No |
| `suspiciousness` | Metric for cheat-likeliness from multiplication of `human_surprise` and `engine_agreement` | Yes |


In [2]:
sf_maia_df = pd.read_csv("moves_sf_maia_features.csv")
print(sf_maia_df.columns)

Index(['game_id', 'ply', 'color_white', 'color_black', 'fen_before',
       'move_san', 'move_uci', 'move_number', 'player_elo', 'opponent_elo',
       'material_balance', 'piece_none', 'piece_pawn', 'piece_knight',
       'piece_bishop', 'piece_rook', 'piece_queen', 'piece_king',
       'captured_none', 'captured_pawn', 'captured_knight', 'captured_bishop',
       'captured_rook', 'captured_queen', 'captured_king', 'is_capture',
       'gives_check', 'is_casteling', 'is_promotion', 'legal_moves_before',
       'source', 'result', 'label', 'sf_cp_loss_log', 'sf_rank_topk',
       'sf_top1', 'elo_group', 'human_prob', 'sf_cp_loss', 'human_surprise',
       'engine_agreement', 'suspiciousness'],
      dtype='object')


In [18]:
sf_maia_df["human_surprise"] = -np.log(sf_maia_df["human_prob"].clip(lower=1e-10)) # add small epsilon to avoid log(0)

sf_maia_df["engine_agreement"] = 1 / (1 + sf_maia_df["sf_cp_loss"].clip(lower=0, upper=1000) / 100)

sf_maia_df["suspiciousness"] = sf_maia_df["human_surprise"] * sf_maia_df["engine_agreement"]

sf_maia_df.to_csv("moves_sf_maia_features.csv", index=False)

Reload dataset

In [3]:
sf_maia_df = pd.read_csv("moves_sf_maia_features.csv")
print(sf_maia_df.columns)


Index(['game_id', 'ply', 'color_white', 'color_black', 'fen_before',
       'move_san', 'move_uci', 'move_number', 'player_elo', 'opponent_elo',
       'material_balance', 'piece_none', 'piece_pawn', 'piece_knight',
       'piece_bishop', 'piece_rook', 'piece_queen', 'piece_king',
       'captured_none', 'captured_pawn', 'captured_knight', 'captured_bishop',
       'captured_rook', 'captured_queen', 'captured_king', 'is_capture',
       'gives_check', 'is_casteling', 'is_promotion', 'legal_moves_before',
       'source', 'result', 'label', 'sf_cp_loss_log', 'sf_rank_topk',
       'sf_top1', 'elo_group', 'human_prob', 'sf_cp_loss', 'human_surprise',
       'engine_agreement', 'suspiciousness'],
      dtype='object')


## Bayesian Logistic Regression

### 1) Crop Features and Split Dataset

In [4]:

FEATURES = [
    "ply",
    "color_white",
    "color_black",
    "move_number",
    "player_elo",
    "opponent_elo",
    "material_balance",
    "piece_none",
    "piece_pawn",
    "piece_knight",
    "piece_bishop",
    "piece_rook",
    "piece_queen",
    "piece_king",
    "captured_none",
    "captured_pawn",
    "captured_knight",
    "captured_bishop",
    "captured_rook",
    "captured_queen",
    "captured_king",
    "is_capture",
    "gives_check",
    "is_casteling",
    "is_promotion",
    "legal_moves_before",
    "sf_cp_loss_log",
    "sf_rank_topk",
    "sf_top1",
    "human_prob",
    "suspiciousness"
]


train_df = sf_maia_df[FEATURES + ["label", "game_id"]].dropna()

X = train_df[FEATURES].values
y = train_df["label"].values

groups = train_df["game_id"].values

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))  #get indices for train-test split based on groups (game_id - same game)
X_train_raw, X_test_raw = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

n_features = X_train_raw.shape[1] # number of features after preprocessing

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)




### 2) Define and Fit Model

In [ ]:
with pm.Model() as model:

    beta0 = pm.Normal("beta0", mu=0, sigma=2)
    beta = pm.Normal("beta", mu=0, sigma=1, shape=n_features)

    eta = beta0 + pm.math.dot(X_train, beta)
    p = pm.math.sigmoid(eta)

    pm.Bernoulli("y_obs", p=p, observed=y_train)

    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=42
    )

Safe trained model 

In [ ]:

az.to_netcdf(trace, "posterior.nc")

joblib.dump(scaler, "scaler.joblib")

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
)

# ============================================================
# 1. Feature definitions
# ============================================================

FEATURES = [
    "ply",
    "color_white",
    "color_black",
    "move_number",
    "player_elo",
    "opponent_elo",
    "material_balance",
    "piece_none",
    "piece_pawn",
    "piece_knight",
    "piece_bishop",
    "piece_rook",
    "piece_queen",
    "piece_king",
    "captured_none",
    "captured_pawn",
    "captured_knight",
    "captured_bishop",
    "captured_rook",
    "captured_queen",
    "captured_king",
    "is_capture",
    "gives_check",
    "is_casteling",
    "is_promotion",
    "legal_moves_before",
    "sf_cp_loss_log",
    "sf_rank_topk",
    "sf_top1",
    "human_prob",
    "suspiciousness",
]

AGG_FEATURES_FINAL = [
    "mean_engine_prob",
    "top5_mean_engine_prob",
    "top10_mean_engine_prob",
    "share_engine_020",
    "share_engine_025",
    "std_engine_prob",
]

# ADVI settings
PLY_ADVI_STEPS = 20_000
GAME_ADVI_STEPS = 20_000
POSTERIOR_SAMPLES = 1000
MINIBATCH_SIZE = 4096
RANDOM_SEED = 42


# ============================================================
# 2. Helper functions
# ============================================================

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def predict_bayes_logreg_mean(
    trace,
    X,
    intercept_name="beta0",
    coef_name="beta",
    chunk_size=50_000,
):
    """
    Predict posterior mean probabilities for Bayesian logistic regression.
    Uses chunks to avoid huge memory usage.
    """
    beta0_samples = trace.posterior[intercept_name].values.reshape(-1)
    beta_samples = trace.posterior[coef_name].values.reshape(-1, X.shape[1])

    probs = np.empty(X.shape[0], dtype=np.float64)

    for start in range(0, X.shape[0], chunk_size):
        end = min(start + chunk_size, X.shape[0])
        X_chunk = X[start:end]

        eta = beta0_samples[:, None] + beta_samples @ X_chunk.T
        p_samples = sigmoid(eta)

        probs[start:end] = p_samples.mean(axis=0)

    return probs


def topk_mean(x, k):
    return x.sort_values(ascending=False).head(k).mean()


def make_game_color_features(df):
    """
    Builds aggregate features per game_id and player_color.
    Requires columns:
    game_id, player_color, engine_prob, y_true_ply
    """
    tmp = df.copy()

    tmp["high_engine_020"] = (tmp["engine_prob"] >= 0.20).astype(int)
    tmp["high_engine_025"] = (tmp["engine_prob"] >= 0.25).astype(int)

    game_color_df = (
        tmp
        .groupby(["game_id", "player_color"])
        .agg(
            mean_engine_prob=("engine_prob", "mean"),
            median_engine_prob=("engine_prob", "median"),
            max_engine_prob=("engine_prob", "max"),
            std_engine_prob=("engine_prob", "std"),
            top5_mean_engine_prob=("engine_prob", lambda x: topk_mean(x, 5)),
            top10_mean_engine_prob=("engine_prob", lambda x: topk_mean(x, 10)),
            share_engine_020=("high_engine_020", "mean"),
            share_engine_025=("high_engine_025", "mean"),
            true_engine_share=("y_true_ply", "mean"),
            true_label=("y_true_ply", "max"),
            n_plies=("engine_prob", "size"),
        )
        .reset_index()
    )

    game_color_df["std_engine_prob"] = game_color_df["std_engine_prob"].fillna(0)

    return game_color_df


def print_binary_metrics(name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n{name}")
    print("Threshold:", threshold)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall:", recall_score(y_true, y_pred, zero_division=0))
    print("F1:", f1_score(y_true, y_pred, zero_division=0))
    print("ROC-AUC:", roc_auc_score(y_true, y_prob))
    print("Brier:", brier_score_loss(y_true, y_prob))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))


def find_best_threshold(y_true, y_prob):
    best = {
        "threshold": None,
        "precision": None,
        "recall": None,
        "f1": -1,
    }

    for threshold in np.arange(0.01, 0.91, 0.01):
        pred = (y_prob >= threshold).astype(int)

        precision = precision_score(y_true, pred, zero_division=0)
        recall = recall_score(y_true, pred, zero_division=0)
        f1 = f1_score(y_true, pred, zero_division=0)

        if f1 > best["f1"]:
            best = {
                "threshold": threshold,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }

    return best


# ============================================================
# 3. Prepare ply-level dataset
# ============================================================

train_df = sf_maia_df[FEATURES + ["label", "game_id"]].dropna().copy()

X = train_df[FEATURES].values.astype("float64")
y = train_df["label"].values.astype("int8")
groups = train_df["game_id"].values

outer_splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=RANDOM_SEED)
train_idx, test_idx = next(outer_splitter.split(X, y, groups))

X_train_raw, X_test_raw = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

ply_scaler = StandardScaler()
X_train = ply_scaler.fit_transform(X_train_raw)
X_test = ply_scaler.transform(X_test_raw)

n_features = X_train.shape[1]

print("Ply train shape:", X_train.shape)
print("Ply test shape:", X_test.shape)


# ============================================================
# 4. Ply-level approximate Bayesian logistic regression
# ============================================================

with pm.Model() as ply_model:
    beta0 = pm.Normal("beta0", mu=0, sigma=2)
    beta = pm.Normal("beta", mu=0, sigma=1, shape=n_features)

    eta = beta0 + pm.math.dot(X_train, beta)
    p = pm.math.sigmoid(eta)

    pm.Bernoulli(
        "y_obs",
        p=p,
        observed=y_train,
    )

    ply_approx = pm.fit(
        n=PLY_ADVI_STEPS,
        method="advi",
        random_seed=RANDOM_SEED,
        progressbar=True,
    )

    ply_trace = ply_approx.sample(POSTERIOR_SAMPLES)

# ============================================================
# 5. Ply-level prediction and evaluation
# ============================================================

y_prob_test = predict_bayes_logreg_mean(
    ply_trace,
    X_test,
    intercept_name="beta0",
    coef_name="beta",
)

print_binary_metrics(
    "PLY LEVEL - Bayesian ADVI Logistic Regression",
    y_test,
    y_prob_test,
    threshold=0.5,
)

best_ply_threshold = find_best_threshold(y_test, y_prob_test)
print("\nBest ply threshold:")
print(best_ply_threshold)


# Also predict train probabilities, needed for game-level training
y_prob_train = predict_bayes_logreg_mean(
    ply_trace,
    X_train,
    intercept_name="beta0",
    coef_name="beta",
)


# ============================================================
# 6. Build train/test result DataFrames with engine_prob
# ============================================================

train_result_df = train_df.iloc[train_idx].copy()
train_result_df["engine_prob"] = y_prob_train
train_result_df["y_true_ply"] = y_train
train_result_df["player_color"] = np.where(
    train_result_df["color_white"] == 1,
    "white",
    "black",
)

test_result_df = train_df.iloc[test_idx].copy()
test_result_df["engine_prob"] = y_prob_test
test_result_df["y_true_ply"] = y_test
test_result_df["player_color"] = np.where(
    test_result_df["color_white"] == 1,
    "white",
    "black",
)


# ============================================================
# 7. Aggregate ply probabilities to game/color features
# ============================================================

game_train_df = make_game_color_features(train_result_df)
game_test_df = make_game_color_features(test_result_df)

Xg_train_raw = game_train_df[AGG_FEATURES_FINAL].values.astype("float64")
Xg_test_raw = game_test_df[AGG_FEATURES_FINAL].values.astype("float64")

yg_train = game_train_df["true_label"].values.astype("int8")
yg_test = game_test_df["true_label"].values.astype("int8")

yg_share_test = game_test_df["true_engine_share"].values.astype("float64")

game_scaler = StandardScaler()
Xg_train = game_scaler.fit_transform(Xg_train_raw)
Xg_test = game_scaler.transform(Xg_test_raw)

n_agg_features = Xg_train.shape[1]

print("\nGame/color train shape:", Xg_train.shape)
print("Game/color test shape:", Xg_test.shape)


# ============================================================
# 8. Game-level approximate Bayesian logistic regression
# ============================================================

with pm.Model() as game_model:
    gamma0 = pm.Normal("gamma0", mu=0, sigma=2)
    gamma = pm.Normal("gamma", mu=0, sigma=1, shape=n_agg_features)

    eta_g = gamma0 + pm.math.dot(Xg_train, gamma)
    p_g = pm.math.sigmoid(eta_g)

    pm.Bernoulli("game_y_obs", p=p_g, observed=yg_train)

    game_approx = pm.fit(
        n=GAME_ADVI_STEPS,
        method="advi",
        random_seed=RANDOM_SEED,
        progressbar=True,
    )

    game_trace = game_approx.sample(POSTERIOR_SAMPLES)


# ============================================================
# 9. Game-level prediction and evaluation
# ============================================================

yg_prob = predict_bayes_logreg_mean(
    game_trace,
    Xg_test,
    intercept_name="gamma0",
    coef_name="gamma",
)

print_binary_metrics(
    "GAME/COLOR LEVEL - Bayesian ADVI Logistic Regression",
    yg_test,
    yg_prob,
    threshold=0.5,
)

best_game_threshold = find_best_threshold(yg_test, yg_prob)
print("\nBest game/color threshold:")
print(best_game_threshold)

# Evaluate how well simple mean ply probability estimates true engine share
predicted_engine_share = game_test_df["mean_engine_prob"].values

print("\nGAME/COLOR LEVEL - mean_engine_prob vs true_engine_share")
print("MAE:", mean_absolute_error(yg_share_test, predicted_engine_share))
print("RMSE:", np.sqrt(mean_squared_error(yg_share_test, predicted_engine_share)))
print(
    "Spearman correlation:",
    pd.Series(yg_share_test).corr(
        pd.Series(predicted_engine_share),
        method="spearman",
    ),
)

# Store predictions in game_test_df for inspection
game_test_df["bayes_game_engine_prob"] = yg_prob

print("\nExamples:")
display(game_test_df.head())

# Posterior coefficient summary for game-level model
gamma_samples = game_trace.posterior["gamma"].values.reshape(-1, n_agg_features)

game_coef_summary = pd.DataFrame({
    "feature": AGG_FEATURES_FINAL,
    "coef_mean": gamma_samples.mean(axis=0),
    "coef_q05": np.quantile(gamma_samples, 0.05, axis=0),
    "coef_q95": np.quantile(gamma_samples, 0.95, axis=0),
})

game_coef_summary["abs_coef_mean"] = game_coef_summary["coef_mean"].abs()

print("\nGame-level coefficient summary:")
display(game_coef_summary.sort_values("abs_coef_mean", ascending=False))

Ply train shape: (1220624, 31)
Ply test shape: (303880, 31)


Output()